# hydromodel Quick Start

This notebook demonstrates the basic usage of hydromodel's unified API for hydrological modeling.

## Key APIs
- `calibrate(config)` — Find optimal model parameters
- `simulate(config)` — Run model with specific parameters
- `evaluate(config, ...)` — Evaluate model performance

In [ ]:
# Verify installation
import hydromodel
print(f"hydromodel version: {hydromodel.__version__}")
print(f"Available models: {list(hydromodel.list_models().keys())[:5]}...")

## Configuration

All APIs use a consistent configuration format with 4 sections:
- `data_cfgs` — Data source and loading
- `model_cfgs` — Model configuration
- `training_cfgs` — Calibration settings
- `evaluation_cfgs` — Evaluation metrics

In [ ]:
# Example configuration for XAJ model calibration
config = {
    "data_cfgs": {
        "dataset": "camels_us",
        "source": "local",
        "basin_ids": ["01013500"],
        "warmup_length": 365,
        "variables": [
            "precipitation",
            "potential_evapotranspiration",
            "streamflow"
        ],
        "train_period": ["1990-01-01", "1995-12-31"],
        "test_period": ["1996-01-01", "2000-12-31"],
    },
    "model_cfgs": {
        "name": "xaj",
        "params": {
            "source_type": "sources",
            "source_book": "HF",
        },
    },
    "training_cfgs": {
        "algorithm": "SCE_UA",
        "SCE_UA": {"rep": 1000, "ngs": 100},
        "loss": "RMSE",
        "output_dir": "results",
        "experiment_name": "example_calibration",
    },
    "evaluation_cfgs": {
        "metrics": ["NSE", "KGE", "RMSE"],
    },
}
print("Configuration ready!")

## Calibration

Find optimal parameters by minimizing the objective function.

In [ ]:
from hydromodel.trainers.unified_calibrate import calibrate

# Run calibration
results = calibrate(config)
print(f"Calibration completed: {len(results)} basins")

## Simulation

Run the model with specific parameter values (no calibration required).

In [ ]:
from hydromodel import simulate

# Add specific parameters to config
config["model_cfgs"]["parameters"] = {
    "K": 0.75, "B": 0.25, "IM": 0.06,
    "UM": 18.0, "LM": 80.0, "DM": 95.0,
    "C": 0.18, "SM": 120.0, "EX": 1.5,
    "KI": 0.35, "KG": 0.45,
    "CS": 0.5, "L": 5.5, "CI": 0.85, "CG": 0.95,
}

# Run simulation
sim_results = simulate(config)
print(f"Simulation keys: {list(sim_results.keys())}")
print(f"qsim shape: {sim_results['simulation']['qsim'].shape}")

## Evaluation

Evaluate model performance on test period.

In [ ]:
from hydromodel.trainers.unified_evaluate import evaluate

# Evaluate on test period
eval_results = evaluate(
    config,
    param_dir="results/example_calibration",
    eval_period="test"
)
print(f"Evaluation completed!")

## Next Steps

- See [Calibration Examples](calibration.md) for detailed calibration workflows
- See [Simulation Examples](simulation.md) for advanced simulation use cases
- See [Flood Events](flood_events.md) for event-based calibration
- See [Data Guide](../data_guide.md) for custom data preparation